<a href="https://colab.research.google.com/github/bonheurjulord-ui/Chatbot-NLP-de-classification-automatique-des-intentions-avec-NLTK-et-Streamlit/blob/main/Chatbot_NLP_de_classification_automatique_des_intentions_avec_NLTK_et_Streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# Installation
!pip install kagglehub nltk scikit-learn pandas numpy -q

In [24]:
import kagglehub
import os
import json
import pandas as pd

# Télécharger dataset
path = kagglehub.dataset_download("niraliivaghani/chatbot-dataset")

print("Path:", path)

# Voir les fichiers
print(os.listdir(path))

Using Colab cache for faster access to the 'chatbot-dataset' dataset.
Path: /kaggle/input/chatbot-dataset
['intents.json']


In [25]:
# Chemin du fichier JSON
file_path = os.path.join(path, os.listdir(path)[0])

# Ouvrir le dataset
with open(file_path, 'r') as file:
    data = json.load(file)

# Aperçu
print(data.keys())

dict_keys(['intents'])


In [26]:
# Voir un exemple
print(data['intents'][0])

{'tag': 'greeting', 'patterns': ['Hi', 'How are you?', 'Is anyone there?', 'Hello', 'Good day', "What's up", 'how are ya', 'heyy', 'whatsup', '??? ??? ??'], 'responses': ['Hello!', 'Good to see you again!', 'Hi there, how can I help?'], 'context_set': ''}


In [27]:
tags = [intent['tag'] for intent in data['intents']]

print("Nombre de catégories :", len(tags))
print(tags)

Nombre de catégories : 38
['greeting', 'goodbye', 'creator', 'name', 'hours', 'number', 'course', 'fees', 'location', 'hostel', 'event', 'document', 'floors', 'syllabus', 'library', 'infrastructure', 'canteen', 'menu', 'placement', 'ithod', 'computerhod', 'extchod', 'principal', 'sem', 'admission', 'scholarship', 'facilities', 'college intake', 'uniform', 'committee', 'random', 'swear', 'vacation', 'sports', 'salutaion', 'task', 'ragging', 'hod']


In [28]:
texts = []
labels = []

for intent in data['intents']:
    tag = intent['tag']

    for pattern in intent['patterns']:
        texts.append(pattern)
        labels.append(tag)

# DataFrame
df = pd.DataFrame({
    'text': texts,
    'label': labels
})

df.head()

,text,label
0,Hi,greeting
1,How are you?,greeting
2,Is anyone there?,greeting
3,Hello,greeting
4,Good day,greeting


In [29]:
print(df.shape)
print(df['label'].value_counts())

(405, 2)
label
course            27
scholarship       26
fees              23
hostel            22
hours             17
creator           16
number            15
library           14
location          14
name              13
salutaion         13
document          13
vacation          12
goodbye           12
canteen           11
sem               11
event             11
greeting          10
ragging           10
placement          9
swear              9
uniform            9
college intake     9
sports             7
floors             7
principal          7
menu               7
syllabus           7
admission          6
committee          6
task               6
facilities         5
ithod              4
computerhod        4
extchod            4
infrastructure     3
random             3
hod                3
Name: count, dtype: int64


In [30]:
##installation et telechargement des ressources
import nltk

# Télécharger ressources NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [31]:
##importer les bibliotheque NLP
import re
import string

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [32]:
##initialiser les outils NLP
# Stopwords anglais
stop_words = set(stopwords.words('english'))

# Lemmatizer
lemmatizer = WordNetLemmatizer()

**Fonction de nettoyage du texte**

Cette fonction va :

✅ mettre en minuscule

✅ enlever la ponctuation

✅ tokeniser

✅ enlever les stopwords

✅ lemmatizer les mots

In [33]:
def preprocess_text(text):

    # Minuscule
    text = text.lower()

    # Enlever ponctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Tokenisation
    tokens = word_tokenize(text)

    # Stopwords + lemmatization
    cleaned_tokens = []

    for word in tokens:
        if word not in stop_words:
            lemma = lemmatizer.lemmatize(word)
            cleaned_tokens.append(lemma)

    return " ".join(cleaned_tokens)

In [34]:
##tester le pretraitement
sample = "Hello! How are you doing today?"

print(preprocess_text(sample))

hello today


In [35]:
##appliquer au dataset
df['clean_text'] = df['text'].apply(preprocess_text)

In [36]:
##afficher le resultat
df.head(10)

,text,label,clean_text
0,Hi,greeting,hi
1,How are you?,greeting,
2,Is anyone there?,greeting,anyone
3,Hello,greeting,hello
4,Good day,greeting,good day
5,What's up,greeting,whats
6,how are ya,greeting,ya
7,heyy,greeting,heyy
8,whatsup,greeting,whatsup
9,??? ??? ??,greeting,


In [37]:
##verifier les catégories du dataset
print("Nombre de catégories :", df['label'].nunique())

print(df['label'].unique())

Nombre de catégories : 38
['greeting' 'goodbye' 'creator' 'name' 'hours' 'number' 'course' 'fees'
 'location' 'hostel' 'event' 'document' 'floors' 'syllabus' 'library'
 'infrastructure' 'canteen' 'menu' 'placement' 'ithod' 'computerhod'
 'extchod' 'principal' 'sem' 'admission' 'scholarship' 'facilities'
 'college intake' 'uniform' 'committee' 'random' 'swear' 'vacation'
 'sports' 'salutaion' 'task' 'ragging' 'hod']


**Vectorisation TF-IDF + préparation ML**

Objectif

Transformer les textes en nombres pour que le modèle puisse apprendre.

In [38]:
##import des outils ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [39]:
##definir X et Y
X = df['clean_text']
y = df['label']

In [40]:
##vectorisation TF-IDF
vectorizer = TfidfVectorizer()

X_tfidf = vectorizer.fit_transform(X)

In [41]:
##Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

**Entraînement du modèle (NLP classifier)**

On va utiliser Naive Bayes, parfait pour NLP.

In [42]:
##import modèle
from sklearn.naive_bayes import MultinomialNB

In [43]:
##entrainement modèle
model = MultinomialNB()

model.fit(X_train, y_train)

MultinomialNB()

In [44]:
##prediction
y_pred = model.predict(X_test)

evalaution modèle

In [45]:
##accuracy
from sklearn.metrics import accuracy_score

print("Accuracy :", accuracy_score(y_test, y_pred))

Accuracy : 0.4691358024691358


In [46]:
##rapport detaillé
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

                precision    recall  f1-score   support

     admission       0.00      0.00      0.00         1
       canteen       0.50      1.00      0.67         1
college intake       0.00      0.00      0.00         2
     committee       0.00      0.00      0.00         2
   computerhod       0.00      0.00      0.00         1
        course       0.44      0.80      0.57         5
       creator       1.00      1.00      1.00         5
      document       1.00      1.00      1.00         2
         event       1.00      1.00      1.00         3
    facilities       0.00      0.00      0.00         1
          fees       1.00      0.60      0.75         5
        floors       0.00      0.00      0.00         1
       goodbye       0.00      0.00      0.00         1
      greeting       0.00      0.00      0.00         3
           hod       0.00      0.00      0.00         1
        hostel       0.33      1.00      0.50         3
         hours       0.25      1.00      0.40  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


CREATION CHATBOT

In [47]:
##fontion chatbot
def chatbot_response(text):

    # prétraitement
    clean = preprocess_text(text)

    # vectorisation
    vector = vectorizer.transform([clean])

    # prédiction
    prediction = model.predict(vector)

    return prediction[0]

In [48]:
##test chatbot_response
print(chatbot_response("Hello"))
print(chatbot_response("I want admission details"))
print(chatbot_response("Where is the library?"))

scholarship
document
library


Amélioration

In [49]:
##probabilité de confiance
def chatbot_response(text):
    clean = preprocess_text(text)
    vector = vectorizer.transform([clean])

    prediction = model.predict(vector)
    proba = model.predict_proba(vector).max()

    return prediction[0], proba

In [ ]:
##chat intéractif
while True:
    msg = input("You: ")

    if msg.lower() == "quit":
        break

    response = chatbot_response(msg)
    print("Bot:", response)

You: hi
Bot: (np.str_('scholarship'), np.float64(0.06580046556648475))


Ce système utilise une approche de traitement du langage naturel basée sur TF-IDF et un classificateur Naive Bayes pour identifier automatiquement l’intention des messages utilisateurs parmi 38 catégories.

**ÉTAPE 8 — Interface Chatbot avec Streamlit + Déploiement Colab (Ngrok)**

Objectif

Créer une interface chatbot accessible via un lien public depuis Colab, grâce à :
- treamlit (interface)
- ngrok (exposition web avec token)

In [57]:
##instaler les dependances
!pip install streamlit pyngrok -q

In [58]:
##sauvegarder le modèle et le vectoriser
import pickle

pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

In [59]:
##créer l'application streamlit
%%writefile app.py
import streamlit as st
import pickle
import nltk
import re

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Charger modèle et vectorizer
model = pickle.load(open("model.pkl", "rb"))
vectorizer = pickle.load(open("vectorizer.pkl", "rb"))

# NLTK setup
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Prétraitement
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)

    clean = []
    for w in tokens:
        if w not in stop_words:
            clean.append(lemmatizer.lemmatize(w))

    return " ".join(clean)

# Prédiction
def predict(text):
    clean = preprocess_text(text)
    vector = vectorizer.transform([clean])
    return model.predict(vector)[0]

# UI
st.set_page_config(page_title="Chatbot NLP", page_icon="🤖")

st.title("🤖 Chatbot NLP - Classification de textes")
st.write("Pose une question et le chatbot la classe automatiquement.")

if "history" not in st.session_state:
    st.session_state.history = []

user_input = st.text_input("Toi :", "")

if user_input:
    response = predict(user_input)

    st.session_state.history.append(("Toi", user_input))
    st.session_state.history.append(("Bot", response))

# Affichage chat
for sender, msg in st.session_state.history:
    if sender == "Toi":
        st.markdown(f"**🧑 {sender}:** {msg}")
    else:
        st.markdown(f"**🤖 {sender}:** {msg}")

Overwriting app.py


In [61]:
from pyngrok import ngrok

# 🔐 Ton token ngrok
NGROK_AUTH_TOKEN = "3CwP5zCjjxtMKBP1zjW2yEJBXIM_6ravauSAomTcfRGHxX6Vr"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [62]:
##lancer streamlit
!streamlit run app.py &>/dev/null &

In [63]:
##créer un tunnel public
public_url = ngrok.connect(8501)
print("🚀 Ton chatbot est accessible ici :", public_url)

🚀 Ton chatbot est accessible ici : NgrokTunnel: "https://reemerge-irritably-grant.ngrok-free.dev" -> "http://localhost:8501"
